In [ ]:
!git clone https://github.com/sukhrobnurali/tooltuned-qwen.git
%cd tooltuned-qwen
!pip install -e ".[colab]" --quiet
!pip uninstall -y torchcodec --quiet

In [ ]:
import sys, os, json
from pathlib import Path
sys.path.insert(0, "/content/tooltuned-qwen/src")
os.makedirs("results/ablations", exist_ok=True)
# Colab exposes user secrets via google.colab.userdata, NOT os.environ --
# the Notebook-access toggle gates userdata.get, but env vars stay empty.
# Pull from userdata first, fall back to env (local runs / Kaggle / CI).
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
assert HF_TOKEN, "Set HF_TOKEN in Colab secrets and toggle Notebook access on"
os.environ["HF_TOKEN"] = HF_TOKEN  # downstream loaders read from env
from tooltuned_qwen.training.train import train
from tooltuned_qwen.training.config import load_config
from tooltuned_qwen.eval.bfcl_holdout import run_bfcl_holdout
print("ready")

In [ ]:
def run_one(config_path: str, *, source_override: str | None = None, n_holdout: int = 50) -> dict:
    cfg = load_config(config_path)
    if source_override is not None:
        cfg = cfg.model_copy(
            update={"data": cfg.data.model_copy(update={"sources": [source_override]})}
        )
    print(f"=== {cfg.run_name} (source={cfg.data.sources}, thinking={cfg.thinking_mode}) ===")
    adapter = train(config=cfg)
    result = run_bfcl_holdout(adapter, n=n_holdout)
    result["run_name"] = cfg.run_name
    result["config_path"] = config_path
    result["sources"] = list(cfg.data.sources)
    result["thinking_mode"] = cfg.thinking_mode
    out = Path("results/ablations") / f"{cfg.run_name}.json"
    out.write_text(json.dumps(result, indent=2, default=str), encoding="utf-8")
    print(f"  -> {result['correct']}/{result['n']} correct ({result['accuracy']:.3f}); saved {out}")
    return result

In [ ]:
# Phase 2.1 -- dataset arm A (xLAM).
xlam_result = run_one("configs/ablation_dataset_xlam.yaml")

In [ ]:
# Phase 2.1 -- dataset arm B (Hermes).
hermes_result = run_one("configs/ablation_dataset_hermes.yaml")

In [ ]:
def _winner(*results: dict) -> dict:
    return max(results, key=lambda r: r["accuracy"])

dataset_winner = _winner(xlam_result, hermes_result)
print("\n=== Phase 2.1 dataset comparison ===")
print(f"xLAM   : {xlam_result['accuracy']:.3f} ({xlam_result['correct']}/{xlam_result['n']})")
print(f"Hermes : {hermes_result['accuracy']:.3f} ({hermes_result['correct']}/{hermes_result['n']})")
print(f"Winner : {dataset_winner['sources'][0]}  -> use this for thinking-mode arms")
WINNER_SOURCE = dataset_winner["sources"][0]

In [ ]:
# Phase 2.2 -- thinking-mode arm (a) preserve.
preserve_result = run_one(
    "configs/ablation_thinking_preserve.yaml", source_override=WINNER_SOURCE
)

In [ ]:
# Phase 2.2 -- thinking-mode arm (b) strip.
strip_result = run_one(
    "configs/ablation_thinking_strip.yaml", source_override=WINNER_SOURCE
)

In [ ]:
# Phase 2.2 -- thinking-mode arm (c) 75/25 mix.
mix_result = run_one(
    "configs/ablation_thinking_mix.yaml", source_override=WINNER_SOURCE
)

In [ ]:
thinking_winner = _winner(preserve_result, strip_result, mix_result)
print("\n=== Phase 2.2 thinking-mode comparison (on " + WINNER_SOURCE + ") ===")
for r in (preserve_result, strip_result, mix_result):
    print(f"{r['thinking_mode']:>10s} : {r['accuracy']:.3f} ({r['correct']}/{r['n']})")
print(f"Winner : {thinking_winner['thinking_mode']}")
print("\nPaste these numbers back to Claude to populate ADRs 0003 + 0004.")